In [ ]:
# ── Cell 1: Imports, environment, and data verification ──────────────────────
from pathlib import Path
import os

import numpy as np
import pandas as pd
import rasterio
from rasterio.crs import CRS
from dotenv import load_dotenv

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT = Path("..").resolve()  # project root (one level above notebooks/)
DATA_RAW = ROOT / "data" / "raw"
ARCHIVE_DIR = DATA_RAW / "archive"
TIFF_DIR = (
    DATA_RAW / "Australia_GISdata_LTAy_AvgDailyTotals_GlobalSolarAtlas-v2_GEOTIFF"
)

# ── Environment variables ─────────────────────────────────────────────────────
load_dotenv(ROOT / ".env")
print("ENV loaded. OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))

# ── 1. Kaggle CSV ─────────────────────────────────────────────────────────────
train_path = ARCHIVE_DIR / "Weather Training Data.csv"
test_path = ARCHIVE_DIR / "Weather Test Data.csv"

df_train = pd.read_csv(train_path, index_col="row ID")
df_test = pd.read_csv(test_path, index_col="row ID")

print(f"\n── Weather Training Data ──")
print(f"  Shape  : {df_train.shape}")
print(f"  Columns: {df_train.columns.tolist()}")
print(
    f"  Locations ({df_train['Location'].nunique()} unique):",
    sorted(df_train["Location"].unique())[:8],
    "...",
)
df_train.head(3)

In [ ]:
# ── 2. Global Solar Atlas GeoTIFFs ────────────────────────────────────────────
# All available bands for Australia
TIFF_BANDS = ["DIF", "DNI", "GHI", "GTI", "OPTA", "PVOUT", "TEMP"]

print("── GeoTIFF file check ──\n")
for band in TIFF_BANDS:
    tif_path = TIFF_DIR / f"{band}.tif"
    with rasterio.open(tif_path) as src:
        print(f"  {band}.tif")
        print(f"    CRS       : {src.crs}")
        print(f"    Bounds    : {src.bounds}")
        print(f"    Resolution: {src.res}  (degrees per pixel)")
        print(f"    Shape     : {src.height} rows × {src.width} cols")
        print(f"    NoData    : {src.nodata}")
        # Quick sanity-read: sample the centre pixel
        cx = src.width // 2
        cy = src.height // 2
        centre_val = list(src.sample([src.xy(cy, cx)]))[0][0]
        print(f"    Centre px : {centre_val:.4f}")
        print()

In [ ]:
# ── Cell 3: Station geocoding (auto) ─────────────────────────────────────────
# Geocodes all station names discovered in the training CSV via Nominatim
# (OpenStreetMap). No coordinates are hardcoded.
#
# Station names are CamelCase (e.g. "CoffsHarbour") — split into words before
# querying so Nominatim can match them. A 1.1 s rate-limit respects OSM ToS.

import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


def camel_to_words(name: str) -> str:
    """'CoffsHarbour' → 'Coffs Harbour', 'PearceRAAF' → 'Pearce RAAF'."""
    return re.sub(r"(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])", " ", name)


locations = sorted(df_train["Location"].unique())

geolocator = Nominatim(user_agent="solar_agent_bom_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1, error_wait_seconds=5)

STATION_COORDS: dict[str, tuple[float, float]] = {}
failed: list[str] = []

print(f"Geocoding {len(locations)} stations via Nominatim (~1 min) …")
for loc in locations:
    query = f"{camel_to_words(loc)}, Australia"
    result = geocode(query)
    if result:
        STATION_COORDS[loc] = (result.latitude, result.longitude)
        print(f"  ✓  {loc:<25}  ({result.latitude:8.4f}, {result.longitude:9.4f})")
    else:
        failed.append(loc)
        print(f"  ✗  {loc:<25}  ← geocoding failed")

if failed:
    print(f"\n⚠ Could not geocode {len(failed)} station(s): {failed}")

print(f"\nGeocoded {len(STATION_COORDS)}/{len(locations)} stations")

# ── Validate coverage against TIFF extent ────────────────────────────────────
TIFF_BOUNDS = dict(lon_min=112.0, lon_max=160.0, lat_min=-44.0, lat_max=-9.0)

df_stations = (
    pd.DataFrame.from_dict(STATION_COORDS, orient="index", columns=["lat", "lon"])
    .rename_axis("Location")
    .reset_index()
)

df_stations["in_bounds"] = df_stations["lon"].between(
    TIFF_BOUNDS["lon_min"], TIFF_BOUNDS["lon_max"]
) & df_stations["lat"].between(TIFF_BOUNDS["lat_min"], TIFF_BOUNDS["lat_max"])

out = df_stations[~df_stations["in_bounds"]]
print(f"\nTotal stations : {len(df_stations)}")
print(f"In TIFF bounds : {df_stations['in_bounds'].sum()}")
print(f"OUT of bounds  : {len(out)}")
if not out.empty:
    print(out[["Location", "lat", "lon"]].to_string(index=False))

# ── Quick sanity plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

rect = mpatches.FancyBboxPatch(
    (TIFF_BOUNDS["lon_min"], TIFF_BOUNDS["lat_min"]),
    TIFF_BOUNDS["lon_max"] - TIFF_BOUNDS["lon_min"],
    TIFF_BOUNDS["lat_max"] - TIFF_BOUNDS["lat_min"],
    boxstyle="square,pad=0",
    linewidth=1.5,
    edgecolor="steelblue",
    facecolor="lightcyan",
    alpha=0.4,
    label="TIFF extent",
)
ax.add_patch(rect)

ok = df_stations[df_stations["in_bounds"]]
oob = df_stations[~df_stations["in_bounds"]]
ax.scatter(ok["lon"], ok["lat"], s=40, color="green", zorder=3, label="In bounds")
ax.scatter(
    oob["lon"],
    oob["lat"],
    s=60,
    color="red",
    marker="x",
    zorder=4,
    label="Out of bounds",
)
for _, row in oob.iterrows():
    ax.annotate(
        row["Location"],
        (row["lon"], row["lat"]),
        fontsize=7,
        xytext=(4, 4),
        textcoords="offset points",
        color="red",
    )

ax.set_xlim(100, 175)
ax.set_ylim(-50, -5)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°S, southern hemisphere)")
ax.set_title("BOM Weather Stations vs. Global Solar Atlas TIFF Extent (auto-geocoded)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: TIFF extraction — monthly PVOUT lookup table (NOT used in model training) ─
# Builds one lookup table for use by the LangGraph agent at inference:
#   df_pvout_monthly — PVOUT[location][month] in kWh/kWp/day (Jan–Dec)
# NorfolkIsland (lon 167.9°E) is outside the TIFF bounds → will be NaN.

MONTH_NAMES = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

coords_xy = [(lon, lat) for lat, lon in df_stations[["lat", "lon"]].values]

# ── Monthly PVOUT (Jan=1 … Dec=12) ───────────────────────────────────────────
monthly_records = {}
MONTHLY_DIR = TIFF_DIR / "monthly"
for month_num in range(1, 13):
    tif_path = MONTHLY_DIR / f"PVOUT_{month_num:02d}.tif"
    with rasterio.open(tif_path) as src:
        sampled = np.array([val[0] for val in src.sample(coords_xy, masked=False)])
        nodata = src.nodata
        if nodata is not None:
            sampled = np.where(sampled == nodata, np.nan, sampled)
    monthly_records[MONTH_NAMES[month_num - 1]] = sampled

df_pvout_monthly = pd.DataFrame(monthly_records, index=df_stations["Location"])
df_pvout_monthly.index.name = "Location"

print("── Monthly PVOUT per station (kWh/kWp/day) ──")
print(df_pvout_monthly.round(3).to_string())

In [ ]:
# ── Cell 5: Validate monthly PVOUT lookup table ───────────────────────────────
# The lookup table is standalone — NOT merged into training data.
# NorfolkIsland is expected to be NaN (outside TIFF bounds at lon 167.9°E).

print("── Lookup table shape ──")
print(f"  df_pvout_monthly: {df_pvout_monthly.shape}  (49 stations × 12 months)")

print("\n── NaN stations (outside TIFF bounds) ──")
nan_stations = df_pvout_monthly[df_pvout_monthly.isna().any(axis=1)].index.tolist()
print(f"  {nan_stations}  ← expected (lon > 160°E)")

print("\n── PVOUT seasonal range across all stations ──")
print(
    f"  Global min : {df_pvout_monthly.min().min():.3f} kWh/kWp/day  ({df_pvout_monthly.min().idxmin()})"
)
print(
    f"  Global max : {df_pvout_monthly.max().max():.3f} kWh/kWp/day  ({df_pvout_monthly.max().idxmax()})"
)
print(f"\n  Monthly averages across all stations:")
print(df_pvout_monthly.mean().round(3).to_string())

# ── Training data uses the raw Kaggle CSVs directly — no solar merge ──────────
print(f"\n── Training data (unmodified Kaggle CSV) ──")
print(f"  df_train : {df_train.shape}")
print(f"  df_test  : {df_test.shape}")
print(f"  All 49 locations retained (incl. NorfolkIsland) for training")

In [ ]:
# ── Cell 6: Feature engineering & preprocessing ───────────────────────────────
# Features are ONLY Kaggle weather columns — no GeoTIFF data enters training.
from sklearn.impute import SimpleImputer

# ── Column taxonomy ───────────────────────────────────────────────────────────
TARGET = "Sunshine"  # continuous hours of sunshine per day → regression target

COMPASS = [
    "N",
    "NNE",
    "NE",
    "ENE",
    "E",
    "ESE",
    "SE",
    "SSE",
    "S",
    "SSW",
    "SW",
    "WSW",
    "W",
    "WNW",
    "NW",
    "NNW",
]

WIND_DIR_COLS = ["WindGustDir", "WindDir9am", "WindDir3pm"]
NUMERIC_COLS = [
    "MinTemp",
    "MaxTemp",
    "Rainfall",
    "Evaporation",
    "WindGustSpeed",
    "WindSpeed9am",
    "WindSpeed3pm",
    "Humidity9am",
    "Humidity3pm",
    "Pressure9am",
    "Pressure3pm",
    "Cloud9am",
    "Cloud3pm",
    "Temp9am",
    "Temp3pm",
]
LABEL_COLS = ["Location"]
# FEATURE_COLS = only weather observations — GeoTIFF bands are NOT included
FEATURE_COLS = NUMERIC_COLS + WIND_DIR_COLS + ["RainToday"] + LABEL_COLS

# ── Working copies ─────────────────────────────────────────────────────────────
tr = df_train.copy()
te = df_test.copy()

# ── 1. Drop rows where TARGET is missing (train only) ────────────────────────
target_nan_before = tr[TARGET].isna().sum()
tr.dropna(subset=[TARGET], inplace=True)
print(f"Dropped {target_nan_before:,} train rows with missing '{TARGET}' target")
print(f"Train after target-NaN drop: {tr.shape}")

# ── 2. Numeric imputation — fit on train, apply to both ───────────────────────
num_imputer = SimpleImputer(strategy="median")
tr[NUMERIC_COLS] = num_imputer.fit_transform(tr[NUMERIC_COLS])
te[NUMERIC_COLS] = num_imputer.transform(te[NUMERIC_COLS])

# ── 3. Wind direction — ordinal encoding via compass order ────────────────────
for col in WIND_DIR_COLS:
    mode_val = tr[col].mode()[0]
    tr[col] = tr[col].fillna(mode_val)
    te[col] = te[col].fillna(mode_val)
    tr[col] = tr[col].map({d: i for i, d in enumerate(COMPASS)}).fillna(-1).astype(int)
    te[col] = te[col].map({d: i for i, d in enumerate(COMPASS)}).fillna(-1).astype(int)

# ── 4. RainToday — binary encode ─────────────────────────────────────────────
for df in (tr, te):
    df["RainToday"] = df["RainToday"].map({"Yes": 1, "No": 0}).fillna(0).astype(int)

# ── 5. Location — label encode ───────────────────────────────────────────────
loc_categories = sorted(tr["Location"].unique())
loc_map = {loc: i for i, loc in enumerate(loc_categories)}
tr["Location"] = tr["Location"].map(loc_map)
te["Location"] = te["Location"].map(loc_map).fillna(-1).astype(int)

# ── 6. Assemble feature matrix ────────────────────────────────────────────────
X_train = tr[FEATURE_COLS].values.astype("float32")
y_train = tr[TARGET].values.astype("float32")
X_test = te[FEATURE_COLS].values.astype("float32")

print(f"\n── Feature matrix (weather-only) ──")
print(f"  X_train : {X_train.shape}   y_train: {y_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"\n  Remaining NaNs in X_train : {np.isnan(X_train).sum()}")
print(f"  Remaining NaNs in X_test  : {np.isnan(X_test).sum()}")
print(
    f"\n  y_train — min: {y_train.min():.2f}  max: {y_train.max():.2f}  "
    f"mean: {y_train.mean():.2f}  std: {y_train.std():.2f}"
)

In [ ]:
# ── Cell 7: XGBoost training ──────────────────────────────────────────────────
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── 1. Hold-out validation split (fit inside train only) ──────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

# ── 2. DMatrix — XGBoost's native data format; preserves NaN for split logic ──
dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=FEATURE_COLS)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=FEATURE_COLS)
dtest = xgb.DMatrix(X_test, feature_names=FEATURE_COLS)

# ── 3. Hyperparameters ────────────────────────────────────────────────────────
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "gamma": 0.1,
    "seed": 42,
    "tree_method": "hist",  # fast histogram algorithm
    "device": "cpu",
}

evals_result = {}
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=30,
    evals_result=evals_result,
    verbose_eval=50,
)

# ── 4. Evaluate on validation set ─────────────────────────────────────────────
y_pred_val = model.predict(dval)
rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)

print(f"\n── Validation metrics (best round: {model.best_iteration}) ──")
print(f"  RMSE : {rmse:.4f} hours")
print(f"  MAE  : {mae:.4f} hours")
print(f"  R²   : {r2:.4f}")

# ── 5. Learning curve ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

train_rmse = evals_result["train"]["rmse"]
val_rmse = evals_result["val"]["rmse"]
rounds = range(len(train_rmse))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rounds, train_rmse, label="Train RMSE", linewidth=1.2)
ax.plot(rounds, val_rmse, label="Val RMSE", linewidth=1.2)
ax.axvline(
    model.best_iteration,
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"Best round ({model.best_iteration})",
)
ax.set_xlabel("Boosting round")
ax.set_ylabel("RMSE (hours of sunshine)")
ax.set_title("XGBoost learning curve — Sunshine prediction")
ax.legend()
plt.tight_layout()
plt.show()

# ── 6. Feature importance ─────────────────────────────────────────────────────
importance = model.get_score(importance_type="gain")
imp_df = (
    pd.Series(importance, name="gain")
    .rename_axis("feature")
    .reset_index()
    .sort_values("gain", ascending=True)
    .tail(15)
)

fig2, ax2 = plt.subplots(figsize=(8, 6))
ax2.barh(imp_df["feature"], imp_df["gain"], color="steelblue")
ax2.set_xlabel("Mean gain")
ax2.set_title("Top-15 feature importances (gain)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8: Save model artifact + inference metadata ─────────────────────────
import json

MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODELS_DIR / "sunshine_predictor.json"
META_PATH = MODELS_DIR / "sunshine_predictor_meta.json"

# ── 1. XGBoost booster → native JSON ─────────────────────────────────────────
model.save_model(str(MODEL_PATH))
print(f"Model saved  → {MODEL_PATH}")

# ── 2. Inference metadata ─────────────────────────────────────────────────────
# Contains:
#   a) Everything needed to reconstruct the feature vector (preprocessing params)
#   b) Monthly PVOUT lookup for the agent's yield formula:
#      daily_yield_kWh = farm_kWp × pvout_monthly[location][month]
#                        × (predicted_sunshine / max_daylight)
meta = {
    # ── model inputs ──────────────────────────────────────────────────────────
    "target": TARGET,
    "feature_cols": FEATURE_COLS,
    "numeric_cols": NUMERIC_COLS,
    "wind_dir_cols": WIND_DIR_COLS,
    "compass": COMPASS,
    "numeric_medians": dict(zip(NUMERIC_COLS, num_imputer.statistics_.tolist())),
    "location_map": loc_map,
    # ── agent yield lookup (GeoTIFF-derived monthly PVOUT, not model features) ─
    "station_coords": {k: list(v) for k, v in STATION_COORDS.items()},
    "station_pvout_monthly": {
        loc: {
            str(m): (
                round(float(df_pvout_monthly.loc[loc, MONTH_NAMES[m - 1]]), 4)
                if not np.isnan(df_pvout_monthly.loc[loc, MONTH_NAMES[m - 1]])
                else None
            )
            for m in range(1, 13)
        }
        for loc in df_pvout_monthly.index
    },
    # ── traceability ──────────────────────────────────────────────────────────
    "val_metrics": {
        "rmse": round(float(rmse), 4),
        "mae": round(float(mae), 4),
        "r2": round(float(r2), 4),
        "best_round": int(model.best_iteration),
    },
}

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved → {META_PATH}")

# ── 3. Verification round-trip ────────────────────────────────────────────────
loaded_model = xgb.Booster()
loaded_model.load_model(str(MODEL_PATH))
y_check = loaded_model.predict(dval)
rmse_check = mean_squared_error(y_val, y_check) ** 0.5
assert abs(rmse_check - rmse) < 1e-4, "Round-trip RMSE mismatch!"
print(f"\nRound-trip verification passed  (RMSE {rmse_check:.4f} == {rmse:.4f})")

# ── 4. Save test predictions ──────────────────────────────────────────────────
PREDS_PATH = MODELS_DIR / "test_predictions.csv"
y_pred_test = model.predict(dtest)
pd.Series(y_pred_test, index=te.index, name="Sunshine_predicted").to_csv(PREDS_PATH)
print(f"Test predictions saved → {PREDS_PATH}  ({len(y_pred_test):,} rows)")

# ── 5. Artifact manifest ──────────────────────────────────────────────────────
print(f"\n── Artifact manifest ──")
for p in sorted(MODELS_DIR.iterdir()):  
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<40}  {size_kb:>8.1f} KB")